# Stage 5 — Explainability Layer

Generates visual explanations for individual predictions using the trained checkpoint:
- **Image side:** Attention Rollout over CLIP's vision transformer (no gradients needed)
- **Text side:** Gradient x Input over token embeddings (single backward pass per example, using the trained classifier head — CLIP's weights are never updated here, only used to trace a gradient path for explanation)

This is the centerpiece for the README and interview talking points — it's what actually demonstrates the XAI angle the project was built around.

In [ ]:
import sys
sys.path.append("..")

import os
import torch
import matplotlib.pyplot as plt
from PIL import Image

from src.data import load_all
from src.features import CLIPFeatureExtractor, get_device
from src.model import FusionClassifierHead
from src.explain import attention_rollout_image, text_token_importance, predict, plot_explanation

device = get_device()
clean_splits, images_root = load_all()
val_df = clean_splits["validation"]

extractor = CLIPFeatureExtractor(attn_implementation="eager")  # needed for Attention Rollout to access attention weights
classifier = FusionClassifierHead().to(device)
classifier.load_state_dict(torch.load("../outputs/checkpoints/best_model.pt", map_location=device))
classifier.eval()

## 1. Pick a mix of examples

We want a mix that makes for a convincing README: at least one correctly-predicted hateful example,
one correctly-predicted not-hateful example, and ideally one the model gets wrong — showing a
failure case honestly is more credible than only showing wins.

In [ ]:
import random
random.seed(7)

# Sample a handful of candidates, then check predictions to categorize them
candidates = val_df.sample(20, random_state=7).reset_index(drop=True)

results = []
for _, row in candidates.iterrows():
    image = Image.open(os.path.join(images_root, row["img"])).convert("RGB")
    image_embeds = extractor.embed_images([image]).to(device)
    text_embeds = extractor.embed_texts([row["text"]]).to(device)
    prob = predict(classifier, image_embeds, text_embeds)
    pred_label = int(prob >= 0.5)
    results.append({
        "img": row["img"], "text": row["text"], "true_label": row["label"],
        "pred_prob": prob, "pred_label": pred_label, "correct": pred_label == row["label"]
    })

import pandas as pd
results_df = pd.DataFrame(results)
print(results_df[["text", "true_label", "pred_prob", "pred_label", "correct"]])

In [ ]:
# Pick 2 correct examples (one of each class) and 1 incorrect example, if available
correct_hateful = results_df[(results_df.correct) & (results_df.true_label == 1)].head(1)
correct_not_hateful = results_df[(results_df.correct) & (results_df.true_label == 0)].head(1)
incorrect = results_df[~results_df.correct].head(1)

selected = pd.concat([correct_hateful, correct_not_hateful, incorrect])
print(f"Selected {len(selected)} examples to explain:")
selected[["text", "true_label", "pred_prob", "correct"]]

## 2. Generate explanations for each selected example

In [ ]:
os.makedirs("../outputs/explanations", exist_ok=True)

for i, row in selected.iterrows():
    image = Image.open(os.path.join(images_root, row["img"])).convert("RGB")
    image_embeds = extractor.embed_images([image]).to(device)

    heatmap = attention_rollout_image(extractor.model, extractor.processor, image, device)
    tokens, scores = text_token_importance(
        extractor.model, extractor.processor, classifier, image_embeds, row["text"], device
    )
    prob = predict(classifier, image_embeds, extractor.embed_texts([row["text"]]).to(device))

    correctness = "correct" if row["correct"] else "INCORRECT"
    save_path = f"../outputs/explanations/example_{i}_{correctness}.png"
    fig = plot_explanation(image, heatmap, tokens, scores, prob, row["true_label"], save_path=save_path)
    plt.show()
    print(f"Saved: {save_path}\n")

## 3. Conclusion

**Image heatmaps (Attention Rollout):** relatively diffuse across all three examples rather than tightly localized on one discriminative region — attention rollout on ViT tends to smear signal broadly across layers when heads are averaged, which is a known limitation of the technique, not a bug in this implementation. Worth stating this limitation explicitly rather than overclaiming sharp localization.

**Text token importance (Gradient x Input):** consistently more informative than the image heatmaps across all three examples:
- Example 1 (correct, not hateful): "they" and "rollin" scored highest — text reads as a song-lyric reference ("they see me rollin, they hatin"), and the model correctly found no clear target of attack.
- Example 2 (**incorrect** — true label hateful, predicted P=0.108): text tokens "them", "country", "kick" were correctly identified as the most important words — genuinely exclusionary language — yet the model still predicted "not hateful" with high confidence. This is the most valuable finding in this notebook: **the text branch found the right signal, but the fused prediction didn't act on it**, likely because the image (a cheering crowd, visually ambiguous/celebratory rather than threatening) pulled the fused representation toward "not hateful." This is a concrete, articulable limitation: the fusion approach can underweight a strong textual signal when the image doesn't visually corroborate it. A natural follow-up (noted as future work) would be an ensemble or gating mechanism that lets a strong unimodal (text-only) signal override an ambiguous fused prediction.
- Example 3 (correct, hateful, P=0.847): "dishwasher", "every", and "the" scored highest, correctly driving a confident hateful prediction — though interestingly the specific sexist framing phrase "real man" scored lower than more neutral surrounding words, suggesting the model may be keying off unusual word co-occurrence patterns rather than directly on the loaded phrase itself.

**Overall takeaway for the README/interview:** the explainability layer doesn't just decorate the predictions — it surfaced a real, specific failure mode (text-image disagreement being resolved in the image's favor) that wouldn't have been visible from the accuracy/AUROC numbers alone. That is precisely the value proposition of building an XAI layer for a trust & safety classifier: not just "what did it predict" but "why did it get this one wrong."

**Next: build the Gradio demo app**, then deploy to Hugging Face Spaces for the live link.